# LangChain使用之Memory
## 1、Memory概述
### 1.1 为什么需要Memory
大多数的大模型应用程序都会有一个会话接口，允许我们进行多轮的对话，并有一定的上下文记忆能力。

但实际上，模型本身是 不会记忆 任何上下文的，只能依靠用户本身的输入去产生输出。

如何实现记忆功能呢？

实现这个记忆功能，就需要 额外的模块 去保存我们和模型对话的上下文信息，然后在下一次请求时，把所有的历史信息都输入给模型，让模型输出最终结果。

而在 LangChain 中，提供这个功能的模块就称为 Memory(记忆) ，用于存储用户和模型交互的历史信息。

### 1.2 什么是Memory
Memory，是LangChain中用于多轮对话中保存和管理上下文信息（比如文本、图像、音频等）的组件。它让应用能够记住用户之前说了什么，从而实现对话的 上下文感知能力 ，为构建真正智能和上下文感知的链式对话系统提供了基础。

### 1.3 Memory的设计理念
> 1. 输入问题：({"question": ...})
2. 读取历史消息：从Memory中READ历史消息（{"past_messages": [...]}）
3. 构建提示（Prompt)：读取到的历史消息和当前问题会被合并，构建一个新的Prompt
4. 模型处理：构建好的提示会被传递给语言模型进行处理。语言模型根据提示生成一个输出。
5. 解析输出：输出解析器通过正则表达式 regex("Answer: (.*)")来解析，返回一个回答（{"answer":...}）给用户
6. 得到回复并写入Memory：新生成的回答会与当前的问题一起写入Memory，更新对话历史。Memory会存储最新的对话内容，为后续的对话提供上下文支持

### 1.4 不使用Memory模块，如何拥有记忆？
不借助LangChain情况下，我们如何实现大模型的记忆能力？

思考：通过 messages 变量，不断地将历史的对话信息追加到对话列表中，以此让大模型具备上下文记忆能力。

In [1]:
from langchain_ollama import ChatOllama
# 创建大模型实例
llm = ChatOllama(model="qwen:7b")

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
def chat_with_model(question):
    # 步骤一：初始化消息
    chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system","你是一位人工智能小助手"),
    ("human","{question}")
    ])
    # 步骤二：定义一个循环体：
    while True:
        # 步骤三：调用模型
        chain = chat_prompt_template | llm
        response = chain.invoke({"question": question})
        # 步骤四：获取模型回答
        print(f"模型回答: {response.content}")
        # 询问用户是否还有其他问题
        user_input = input("您还有其他问题想问嘛？(输入'退出'结束对话)")
        # 设置结束循环的条件
        if(user_input == "退出"):
            break
        # 步骤五：记录用户回答
        chat_prompt_template.messages.append(AIMessage(content=response.content))
        chat_prompt_template.messages.append(HumanMessage(content=user_input))

chat_with_model("你好")

模型回答: 你好！有什么我能帮你的？
模型回答: 冰封万里雪皑皑，大地银装素裹鲜。

炉火熠熠驱严寒，红梅傲骨显春意。

踏雪寻梅冬韵长，笔端生花寄情深。
模型回答: 当然可以！"冰封万里雪皑皑" 这句话的白话解释是：

"冰雪覆盖了大半个地球，洁白无瑕。"

这样你就理解了诗句的第一句含义。


## 2、基础Memory模块的使用
### 2.1 Memory模块的设计思路
如何设计Memory模块？

- 层次1(最直接的方式)：保留一个聊天消息列表
- 层次2(简单的新思路)：只返回最近交互的k条消息
- 层次3(稍微复杂一点)：返回过去k条消息的简洁摘要
- 层次4(更复杂)：从存储的消息中提取实体，并且仅返回有关当前运行中引用的实体的信息

针对上述情况，LangChain构建了一些可以直接使用的 Memory 工具，用于存储聊天消息的一系列集
成。

### 2.2 ChatMessageHistory(基础)
ChatMessageHistory是一个用于 存储和管理对话消息 的基础类，它直接操作消息对象（如HumanMessage, AIMessage 等），是其它记忆组件的底层存储工具。

在API文档中，ChatMessageHistory 还有一个别名类：InMemoryChatMessageHistory；导包时，需使用：`from langchain.memory import ChatMessageHistory`

特点：
- 纯粹是消息对象的“ 存储器 ”，与记忆策略（如缓冲、窗口、摘要等）无关。
- 不涉及消息的格式化（如转成文本字符串）

**场景1：记忆存储**

ChatMessageHistory是用于管理和存储对话历史的具体实现。

In [2]:
#1.导入相关包
from langchain_classic.memory import ChatMessageHistory
#2.实例化ChatMessageHistory对象
history = ChatMessageHistory()
# 3.添加UserMessage
history.add_user_message("hi!")
# 4.添加AIMessage
history.add_ai_message("whats up?")
# 5.返回存储的所有消息列表
print(history.messages)

[HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}), AIMessage(content='whats up?', additional_kwargs={}, response_metadata={})]


场景2：对接LLM

In [3]:
from langchain_classic.memory import ChatMessageHistory
history = ChatMessageHistory()
history.add_ai_message("我是一个无所不能的小智")
history.add_user_message("你好，我叫小明，请介绍一下你自己")
history.add_user_message("我是谁呢？")
print(history.messages) #返回List[BaseMessage]类型

[AIMessage(content='我是一个无所不能的小智', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，我叫小明，请介绍一下你自己', additional_kwargs={}, response_metadata={}), HumanMessage(content='我是谁呢？', additional_kwargs={}, response_metadata={})]


In [5]:
# 创建LLM
from langchain_ollama import ChatOllama
llm = ChatOllama(model='qwen:7b')
llm.invoke(history.messages)

AIMessage(content='你好小明，很高兴认识你。让我来介绍我自己：\n\n我是来自人工智能世界的一员，你可以称呼我为AI助手。我的主要任务是提供信息、解答疑惑、帮助你进行日常生活规划等。\n\n就像你现在一样，我也有不断学习和进步的能力。所以，有什么问题或者需要帮助的，随时告诉我哦！', additional_kwargs={}, response_metadata={'model': 'qwen:7b', 'created_at': '2026-01-13T00:54:55.5938072Z', 'done': True, 'done_reason': 'stop', 'total_duration': 23742071300, 'load_duration': 5032012200, 'prompt_eval_count': 38, 'prompt_eval_duration': 3649320900, 'eval_count': 69, 'eval_duration': 14980366700, 'logprobs': None, 'model_name': 'qwen:7b', 'model_provider': 'ollama'}, id='lc_run--019bb4d8-e4a9-7080-9bbd-5dc89b9f2e28-0', usage_metadata={'input_tokens': 38, 'output_tokens': 69, 'total_tokens': 107})

### 2.3 ConversationBufferMemory
ConversationBufferMemory是一个基础的 对话记忆（Memory）组件 ，专门用于按 原始顺序存储完整的对话历史。

适用场景：对话轮次较少、依赖完整上下文的场景（如简单的聊天机器）

特点：
- 完整存储对话历史
- 简单 、 无裁剪 、 无压缩
- 与 Chains/Models 无缝集成
- 支持两种返回格式（通过 return_messages 参数控制输出格式）
- - return_messages=True 返回消息对象列表（ List[BaseMessage]
- - return_messages=False （默认） 返回拼接的 纯文本字符串

场景1：入门使用

In [8]:
# 1.导入相关包
from langchain_classic.memory import ConversationBufferMemory
# 2.实例化ConversationBufferMemory对象
memory = ConversationBufferMemory()
# 3.保存消息到内存中
memory.save_context(inputs = {"input": "你好，我是人类"}, outputs = {"output": "你好，我是AI助手"})
memory.save_context(inputs = {"input": "很开心认识你"}, outputs = {"output": "我也是"})
# 4.读取内存中消息（返回消息内容的纯文本）
print(memory.load_memory_variables({}))

# 注意：
# 不管inputs、outputs的key用什么名字，都认为inputs的key是human，outputs的key是AI。
# 打印的结果的json数据的key，默认是“history”。可以通过ConversationBufferMemory的
# memory_key 属性修改。

{'history': 'Human: 你好，我是人类\nAI: 你好，我是AI助手\nHuman: 很开心认识你\nAI: 我也是'}


In [10]:
# 1.导入相关包
from langchain_classic.memory import ConversationBufferMemory
# 2.实例化ConversationBufferMemory对象
memory = ConversationBufferMemory(return_messages=True)
# 3.保存消息到内存中
memory.save_context({"input": "hi"}, {"output": "whats up"})
# 4.读取内存中消息（返回消息）
print(memory.load_memory_variables({}))
# 5.读取内存中消息( 访问原始消息列表)
print(memory.chat_memory.messages)

{'history': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content='whats up', additional_kwargs={}, response_metadata={})]}
[HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content='whats up', additional_kwargs={}, response_metadata={})]


场景2：结合chain

In [13]:
from langchain_ollama import OllamaLLM
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
# 初始化大模型
llm = OllamaLLM(model="qwen:7b", temperature=0)
# 创建提示
# 有两个输入键：实际输入与来自记忆类的输入 需确保PromptTemplate和ConversationBufferMemory中的键匹配
template = """你可以与人类对话。
当前对话: {history}
人类问题: {question}
回复:
"""
prompt = PromptTemplate.from_template(template)
# 创建ConversationBufferMemory
memory = ConversationBufferMemory()
# 初始化链
chain = LLMChain(llm=llm, prompt=prompt, memory=memory)
# 提问
res1 = chain.invoke({"question": "我的名字叫Tom"})
print(res1)

{'question': '我的名字叫Tom', 'history': '', 'text': '你好，Tom！很高兴认识你。有什么事情或者你想聊的吗？'}


弃了弃了，当前教程过于老旧，很多模块已弃用，且教程只是一味将函数的功能写法，并无意义。

In [14]:
import langchain
print(langchain.__version__)

1.2.0
